# Прогноз расходов территорий: seasonal naive и LightGBM

**Задача:** прогнозировать `total_spending` по каждой территории на 1, 3, 6 и 12 месяцев вперёд.
**Данные:** 24 месяца (2023–2024), 2190 территорий. Это очень мало: в данных всего один годовой цикл,
и это определяет почти все решения ниже.
**План:** честная схема проверки → простые базовые модели → LightGBM → выбор итоговой модели.

In [1]:
%load_ext autoreload
%autoreload 2
from forecasting import *

df_general = pd.read_parquet('../data/df_general.parquet')

dates_all = pd.to_datetime(np.sort(df_general['date'].unique()))
EVAL_TARGETS = list(dates_all[-6:]) # последние 6 месяцев
holiday = df_general.groupby('date')['holiday_count'].first()

rng = np.random.default_rng(42)
sample_ids = rng.choice(df_general['territory_id'].unique(), 300, replace=False)  # для сравнения с Prophet

## Схема проверки

- Для каждого целевого месяца модель обучается **только на данных до месяца прогноза** (никакого подглядывания в будущее).
- Все модели проверяются на **одних и тех же последних 6 целевых месяцах**. В первой версии у горизонтов
  были разные целевые месяцы, и ошибки на разных горизонтах нельзя было сравнивать.
- Метрика: MAE в рублях (метрика конкурса). Дополнительно смотрю WAPE и R².
- Горизонт 12 мес проверяется на одной точке (декабрь 2024): больше данных нет. Это ограничение,
  а не особенность модели.

In [2]:
MODELS_BASE = {
    'naive': (naive_forecast, {}),
    'seasonal_naive': (seasonal_naive_forecast, {}),
}
summarize(MODELS_BASE, df_general, EVAL_TARGETS)

,1,3,6,12
naive,2023.0,2479.0,3528.0,4287.0
seasonal_naive,1135.0,1179.0,1553.0,4278.0


### Naive и seasonal naive

- **Naive** — прогноз равен последнему известному значению.
- **Seasonal naive** — значение того же месяца год назад, умноженное на медианный годовой рост по стране.
  Идея: у расходов сильная годовая сезонность (например, декабрьский пик), а у naive её нет.

| MAE | 1 мес | 3 мес | 6 мес | 12 мес |
|---|---|---|---|---|
| naive | 2023 | 2479 | 3528 | 4287 |
| seasonal naive | 1135 | 1179 | 1553 | 4278 |

**Вывод:** seasonal naive почти вдвое точнее naive на коротких горизонтах. Это главная планка,
которую должны обойти остальные модели. (На 12 мес они совпадают: «год назад до цели» там —
это ровно момент разреза, а годовой рост без ещё одного года истории оценить нельзя.)

---

## LightGBM

**Один глобальный LightGBM** на все территории (а не по модели на территорию): у каждой территории
всего 12–23 точки, а вместе их ~50 тысяч. Модель учится на похожести территорий и общей сезонности.

**Как устроено (схема direct):** для горизонта h признаки берутся на момент T, цель — значение в T+h.
Для каждого горизонта своя модель. В первой версии признаки при обучении и при прогнозе считались по-разному
(лаги и скользящие средние), и результаты были недостоверны. Исправлено.

**Цель — относительное изменение** `y(T+h)/y(T) − 1`, а не уровень: масштабы территорий отличаются
в 8 раз (11–91 тысяча), на относительном изменении модели проще.

**Функция потерь L1 с весом `y(T)`:** эквивалентна MAE в рублях, то есть оптимизируется метрика конкурса.

**Признаки:** динамика (`r_1..r_6`), скользящие средние, «тот же месяц год назад» (`ly`), месяц цели
(sin/cos), праздники целевого месяца (календарь известен заранее), макро и структура расходов на момент T,
регион и тип района как категории.
**Убрано:** `territory_id` как число (порядок номеров не несёт смысла) и `share_*`/`national_spending` за месяц
цели (в реальном прогнозе их ещё нет, это была утечка).

In [3]:
MODELS_LGBM = {
    'lgbm_direct': (lgbm_direct_forecast, {'holiday': holiday}),
    'blend_50_50': (blend_forecast, {'holiday': holiday, 'w': 0.5}),
}
summarize(MODELS_LGBM, df_general, EVAL_TARGETS)

,1,3,6,12
lgbm_direct,1434.0,1408.0,2744.0,4278.0
blend_50_50,1041.0,1051.0,1958.0,4278.0


| MAE | 1 мес | 3 мес | 6 мес | 12 мес |
|---|---|---|---|---|
| seasonal naive | 1135 | 1179 | 1553 | 4278 |
| lgbm_direct | 1434 | 1408 | 2744 | 4278 |
| blend 50/50 | 1041 | 1051 | 1958 | 4278 |

**Вывод:** чистый LightGBM хуже seasonal naive на всех горизонтах, но их смесь лучше обеих моделей
на 1 и 3 мес. Значит, модель ловит то, чего нет у seasonal naive, но сама недообучена.
На 12 мес обучающих пар (T, T+12) нет, поэтому там везде откат на seasonal naive.

### Эксперимент: LightGBM как поправка к «году назад» (не помог)

Идея: пусть модель предсказывает только отклонение от seasonal naive, тогда ей нужно выучить меньше.
Проблема: для обучения нужна база «год назад», а на ранних разрезах таких пар слишком мало.

## Подбор веса смеси

Прогнозы собираются один раз (`collect_preds`), а смешиваются потом с разными весами, не пересчитывая модели.
Вес — доля LightGBM, остальное — seasonal naive.
**Осторожно:** вес подбирается на тех же месяцах, где измеряется ошибка, поэтому на новых данных
результат может быть чуть хуже. Я зафиксировала вес 0,25 и дальше его не подгоняю.

In [3]:
fns = {'sn': (seasonal_naive_forecast, {}),
       'lgb': (lgbm_direct_forecast, {'holiday': holiday}),
       'res': (lgbm_resid_forecast, {'holiday': holiday})}

def mae_by_month(pred, P):
    return (pred - P['y']).abs().groupby(P['target']).mean().mean()

rows = []
for h in [1, 3, 6]:
    P = collect_preds(df_general, h, fns, EVAL_TARGETS)
    rows.append({'h': h, 'model': 'sn', 'w': 0.0, 'MAE': mae_by_month(P['sn'], P)})
    for model in ['lgb', 'res']:
        for w in [0.25, 0.5, 0.75, 1.0]:
            rows.append({'h': h, 'model': model, 'w': w,
                         'MAE': mae_by_month(w * P[model] + (1 - w) * P['sn'], P)})
print(pd.DataFrame(rows).pivot_table(index=['model', 'w'], columns='h', values='MAE').round(0))

h=1 lgb: нет прогноза у 0.0% строк (подставлен sn)
h=1 res: нет прогноза у 1.9% строк (подставлен sn)
h=3 lgb: нет прогноза у 0.0% строк (подставлен sn)
h=3 res: нет прогноза у 1.6% строк (подставлен sn)
h=6 lgb: нет прогноза у 0.0% строк (подставлен sn)
h=6 res: нет прогноза у 1.2% строк (подставлен sn)
h                1       3       6
model w                           
lgb   0.25  1012.0  1038.0  1685.0
      0.50  1041.0  1051.0  1958.0
      0.75  1198.0  1190.0  2322.0
      1.00  1434.0  1408.0  2744.0
res   0.25  1072.0  1150.0  1596.0
      0.50  1069.0  1221.0  1687.0
      0.75  1125.0  1378.0  1814.0
      1.00  1233.0  1596.0  1973.0
sn    0.00  1135.0  1179.0  1553.0


| MAE | 1 мес | 3 мес | 6 мес |
|---|---|---|---|
| seasonal naive (w=0) | 1135 | 1179 | 1553 |
| lgbm, w=0.25 | **1012** | **1038** | 1685 |
| lgbm, w=0.5 | 1041 | 1051 | 1958 |
| lgbm, w=1.0 | 1434 | 1408 | 2744 |
| поправка (res), лучший вес | 1069 | 1150 | 1596 |

**Выводы:**
- Малая доля LightGBM (25–50%) даёт −11–12% к seasonal naive на 1 и 3 мес; разница между 0,25 и 0,5 мала,
  то есть результат не зависит от точного значения веса.
- На 6 мес любая доля LightGBM ухудшает результат. Причина — мало обучающих данных для такого горизонта.
- Вариант «поправка к году назад» хуже обычного LightGBM, оставляю его как негативный результат.

## Горизонт 12 месяцев: главная ошибка — пропущенный рост

На 12 мес все модели дают MAE около 4300, потому что не знают годового роста расходов
(данных за предыдущий год у нас нет). Проверка: как менялась бы ошибка, если бы рост был известен?

| рост | 0% | 5% | 8% | 10% | 12% | **15%** | 18% |
|---|---|---|---|---|---|---|---|
| MAE | 4688 | 3131 | 2283 | 1788 | 1393 | **1124** | 1472 |

Минимум около 15% совпадает с фактическим годовым ростом (медиана YoY за декабрь 2024 — 1,147).
**Но это «оракульная» оценка:** годовой рост нельзя посчитать внутри выборки. Поэтому для 12 мес
я использую сценарное допущение: `последнее значение × (1 + 0,09)`, а не «выученный» рост.
Результат на 12 мес — одна точка, его надо считать предварительным.

## Итоговая модель

- 1 и 3 мес: 25% LightGBM + 75% seasonal naive;
- 6 мес: seasonal naive (LightGBM не помогает);
- 12 мес: последнее значение × 1,09.

In [5]:
MODELS_FINAL = {
    'naive': (naive_forecast, {}),
    'seasonal_naive': (seasonal_naive_forecast, {}),
    'final_forecast': (final_forecast, {'holiday': holiday}),
}
print('Все 2190 территорий:')
display(summarize(MODELS_FINAL, df_general, EVAL_TARGETS))
print('300 территорий (как для Prophet):')
display(summarize(MODELS_FINAL, df_general, EVAL_TARGETS, eval_ids=sample_ids))

Все 2190 территорий:


,1,3,6,12
naive,2023.0,2479.0,3528.0,4287.0
seasonal_naive,1135.0,1179.0,1553.0,4278.0
final_forecast,1012.0,1038.0,1553.0,1962.0


300 территорий (как для Prophet):


,1,3,6,12
naive,1935.0,2353.0,3364.0,4069.0
seasonal_naive,1113.0,1140.0,1508.0,4058.0
final_forecast,990.0,1009.0,1508.0,1886.0


## Выводы

| MAE, 300 территорий | 1 мес | 3 мес | 6 мес | 12 мес |
|---|---|---|---|---|
| Prophet | 1789 | 2004 | 2348 | 13730 |
| seasonal naive | 1113 | 1140 | 1508 | 4058 |
| **итоговая модель** | **990** | **1009** | **1508** | **1886** |

(Chronos, на всех территориях: 2190 / 2898 / 4991 / 6177. Prophet и Chronos считались в своих ноутбуках.)

- Простая сезонная модель уже сильно лучше Prophet и Chronos: на 24 месяцах им трудно «увидеть» годовую сезонность.
- Глобальный LightGBM добавляет около 11% на 1 и 3 мес и не помогает на 6.
- На 12 мес ошибка определяется годовым ростом, а не моделью, оценка по одной точке.
- Параметры (вес 0,25 и рост 0,09) подобраны на тех же месяцах, на которых мерится ошибка.